# Toybox batch 1 — 60 questions, `openai/gpt-6-luna`

Same format and prompt as the single pilot question (notebook 53): 3 before/after demonstrations of 3 different
objects, 1 query image of a 4th object, 4 candidates, "Answer with exactly one letter".

| task | n | transformation | candidates |
|---|---|---|---|
| `R` rotation | 10 | 90° counter-clockwise, 90° clockwise or 180°, from recorded frames | 0 / 90 / 180 / 270° |
| `T` translation | 10 | left / right / up / down (crop-window shift = camera pan) | the 4 directions |
| `RT` rotation + translation | 10 | ±90° rotation (recorded) plus a shift | correct, rotation only, translation only, wrong rotation direction + shift |
| `M` mirror | 10 | left-right mirror (digital flip) about the object's own axis | correct, top-bottom mirror, rotate 180°, unchanged |
| `RM` rotation then mirror | 10 | ±90° rotation (recorded) then left-right mirror | correct, rotation only, mirror only, wrong order |
| `TM` translation + mirror | 10 | left-right mirror plus a shift | correct, mirror only, shift only, top-bottom mirror + shift |

The questions are already in the repository (`experiments/toybox_rotation_pilot/batch1/`). Answers live in
`batch1/answers.json` and are only read when scoring. One call per question (60 calls, about 5 cents). Re-running
skips questions that already have a successful response.

Needs the Colab secrets `GITHUB_TOKEN` and `OPENROUTER_API_KEY` (never printed).


In [ ]:
# --- 2. Bootstrap: locate or clone the PRIVATE repository (credentials are never printed or stored) ---
import base64, importlib.util, os, pathlib, subprocess, sys

REPO_URL = "https://github.com/zhexuandliu/Decompositionality.git"
REPO_BRANCH = "main"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

def _find_repo():
    candidates = [pathlib.Path("/content/Decompositionality"), pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    for p in candidates:
        if (p / "src" / "decompositionality").is_dir():
            return p.resolve()
    return None

REPO_DIR = _find_repo()
if REPO_DIR is None:
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
    if not token:
        import getpass
        token = getpass.getpass("GitHub token with read access to the private repo (input hidden): ")
    # Pass the credential through environment-scoped git config (not the URL, not the command line,
    # not persisted in .git/config).
    _hdr = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
    _env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="1",
                GIT_CONFIG_KEY_0="http.https://github.com/.extraheader", GIT_CONFIG_VALUE_0=_hdr)
    REPO_DIR = pathlib.Path("/content/Decompositionality" if IN_COLAB else "Decompositionality").resolve()
    _r = subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
                        env=_env, capture_output=True, text=True)
    del token, _hdr, _env
    if _r.returncode != 0:
        raise RuntimeError("git clone failed (check the GITHUB_TOKEN secret / repo access):\n" + _r.stderr[-2000:])

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
for _mod in ("numpy", "pandas", "matplotlib", "torch"):
    if importlib.util.find_spec(_mod) is None:  # Colab already ships these; never reinstall torch/CUDA here
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _mod], check=True)
REPO_COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Repository:", REPO_DIR, "| commit:", REPO_COMMIT or "unknown")


In [ ]:
# --- 3. Update the repository (the question files were added recently) -------------------------------------------
import base64, subprocess
_tok = None
if IN_COLAB:
    from google.colab import userdata
    _tok = userdata.get("GITHUB_TOKEN")
_env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
if _tok:
    _env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                GIT_CONFIG_VALUE_0="AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{_tok}".encode()).decode())
_r = subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"], env=_env, capture_output=True, text=True)
del _tok, _env
print(_r.stdout[-500:] or _r.stderr[-500:])
PILOT = REPO_DIR / "experiments" / "toybox_rotation_pilot"
assert (PILOT / "batch1" / "answers.json").exists(), "batch files missing: pull failed?"
print("batch folder:", PILOT / "batch1")


In [ ]:
# --- 4. One example question (composites are for viewing only; they are not sent) -----------------------------
from IPython.display import Image as _Img, display
BATCH = PILOT / "batch1"
for qid in ["R00", "T00", "RT00", "M00", "RM00", "TM00"]:
    print(qid); display(_Img(filename=str(BATCH / qid / "composite.jpg"), width=900))


In [ ]:
# --- 5. Calls: one per question (resumable) ------------------------------------------------------------------------
if IN_COLAB and not os.environ.get("OPENROUTER_API_KEY"):
    from google.colab import userdata
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")      # never printed
MODEL = "openai/gpt-6-luna"
r = subprocess.run([sys.executable, str(PILOT / "eval_batch.py"), "--batch", str(BATCH), "--model", MODEL, "--workers", "8"],
                   capture_output=True, text=True, cwd=str(PILOT))
print(r.stdout[-6000:]); print(r.stderr[-3000:])


In [ ]:
# --- 6. Results and download -------------------------------------------------------------------------------------
import pandas as pd, shutil
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 60)
df = pd.read_csv(BATCH / f"scores_{MODEL.replace('/', '_')}.csv")
print(df.groupby("task").correct.agg(["mean", "sum", "size"]))
print("\nWhat was chosen when wrong:"); print(df[~df.correct].groupby(["task", "chosen_kind"]).size())
display(df[["qid", "task", "rule", "answer", "choice", "correct", "chosen_kind", "reasoning_tokens", "cost"]])
print("total cost $", round(pd.to_numeric(df.cost, errors="coerce").sum(), 4))
if IN_COLAB:
    from google.colab import files
    out = "/content/toybox_batch1_results"
    os.makedirs(out, exist_ok=True)
    for f in BATCH.glob("*_" + MODEL.replace("/", "_") + ".*"):
        shutil.copy(f, out)
    shutil.make_archive(out, "zip", out)
    files.download(out + ".zip")
